In [1]:
from langchain_milvus import Milvus
from langchain_ollama import OllamaEmbeddings

"""
    创建向量化模型
"""
ollama_embeddings = OllamaEmbeddings(
    model="qwen3-embedding:0.6b", dimensions=1024
)

In [2]:
from app.core.config import settings
from langchain_milvus import BM25BuiltInFunction

"""
    创建langchain包装过的向量数据库
"""

vector_store = Milvus(
     # 稠密向量模型
    embedding_function=ollama_embeddings,
    # collection名称
    collection_name="langchain_collection",
    # 做稀疏向量的操作函数
    builtin_function=BM25BuiltInFunction(
        # 指定中文分词
        analyzer_params={"type": "chinese"}
    ),
     # 向量字段命名
    vector_field=["dense", "sparse"],
    # 数据库连接地址
    connection_args={
        "uri": settings.rag.milvus_url,
    },
    # 是否删除旧的collection，避免重复创建
    drop_old=False,
    # 自动主键
    auto_id=True
)


In [3]:
"""
    定义自定义reranker函数
"""
from pymilvus import Function, FunctionType
def create_cross_encoder_ranker(queries: list[str]):
    return Function(
        # 重排函数名
         name="小汪重排ranker",
        input_field_names=["text"],  # 原始文档字段
        function_type=FunctionType.RERANK,  # ranker类型，这里是固定值
        params={
            # 使用模型进行reranker
            "reranker": "model",
            "provider": "ali",  # rerank模型提供者
            "model_name": "gte-rerank-v2",  # rerank模型名称
            "queries": queries,  # 查询条件
            "max_client_batch_size": 5,  # 向模型发送请求时的批处理限制
        },
    )

In [4]:
user_input = '孔子是谁'

result = vector_store.similarity_search_with_score(
    query=user_input,
    # 最终返回的3条
    k=3,
    # 初筛召回的文档数量
    fetch_k=5,
    # expr=filter
    reranker=create_cross_encoder_ranker([user_input])
)

for document, score in result:
    print(f'文档得分:{score}')
    print(f'文档内容:{document.model_dump_json(indent=2)}')

文档得分:0.1468632072210312
文档内容:{
  "id": null,
  "metadata": {
    "H1": "第一章 教育概述",
    "H2": "第一节 中外教育家及其教育思想",
    "H3": "（三）孔子及《论语》主要思想",
    "pk": 468850925521564766
  },
  "page_content": "### （三）孔子及《论语》主要思想  \n**教育作用**：庶、富、教；性相近，习相远。  \n**教育对象**：“有教无类”，教育民主思想。  \n**教育目的**：以完善人格为教育的首要目的，培养士和君子。  \n**教学内容**：文、行、忠、义。  \n**教学过程**：学、思、习、行。  \n**教学原则**：因材施教、启发诱导、学思结合、谦虚笃实。  \n**教师观**：其身正，不令而行；其身不正，虽令不从。",
  "type": "Document"
}
文档得分:0.09249099344015121
文档内容:{
  "id": null,
  "metadata": {
    "H1": "第一章 教育概述",
    "H2": "第一节 中外教育家及其教育思想",
    "H3": "（五）荀子主要思想",
    "pk": 468850925521564768
  },
  "page_content": "### （五）荀子主要思想  \n**人性论**：性恶论，化性起伪，善德是后天习得的，重视教育的作用。  \n**教学原则**：学以致用，锲而不舍。",
  "type": "Document"
}
文档得分:0.0800798162817955
文档内容:{
  "id": null,
  "metadata": {
    "H1": "第一章 教育概述",
    "H2": "第一节 中外教育家及其教育思想",
    "H3": "（四）孟子主要思想",
    "pk": 468850925521564767
  },
  "page_content": "### （四）孟子主要思想  \n**人性论**：人性本善，人先天具有仁、义、礼、智四个“善端”。  \n**教育作用**：发扬善端，培养道德完人，得天下英才而教育之。  \n**教学